In [1]:
def filter_wwt(wwt: pd.DataFrame, split:str):
    """Filter the WWT to packet or flow data. Split must be either 'Flow' or 'Packet'."""
    if split.lower() == 'packet':
        pkt_features = [feat for feat in wwt.columns if feat.startswith('pkt_')]
        return wwt[pkt_features]
    if split.lower() == 'flow':
        flow_features = [feat for feat in wwt.columns if feat.startswith('flow_')]
        return wwt[flow_features]
    else:
        raise ValueError("Split must be one of 'Flow' or 'Packet'")

X_train, X_val, X_test, y_train, y_val, y_test = get_training_data()
#train phase 2 models
print('\nFilter flow features out of training set for anomoly detectors.')
X_train_pkt = filter_wwt(X_train, split='Packet')
X_val_pkt = filter_wwt(X_val, split='Packet')
trained_models = train_all(X_train_pkt, X_val_pkt, y_train, y_val)
#---------------------
#get_train for phase 3
#X_train_flow, X_val_flow, y_train_flow, y_val_flow = get_training_data()(from phase3_1.ipynb)
#train phase 3 model here
#---------------------
print('Filter flow features out of testing for unsupervised anomoly detection.')
X_test_pkt = filter_wwt(X_test, split='Packet')

print('\nAnomoly detection on testing data')
X_arr_pkt, score_arr_pkt, label_arr_pkt = predict_phase2(X_test_pkt)

print('\nAsserting that X_test_pkt and X_arr_pkt are the same to confirm that predict_phase2 has not changed order')
np.testing.assert_allclose(X_test_pkt.values, X_arr_pkt)

print(f"\nLabel the full testing set with anomoly detection labels. \nX_test shape: {X_test.shape}. label_arr_pkt shape: {label_arr_pkt.shape}")
X_test['phase_2_label'] = label_arr_pkt.ravel()
from notebooks.preprocessing.phase2to3connection import filter_drop

print('\nFilter out non-anomolous rows, and remove the anomoly detection labels')
filtered_X = filter_drop(df=X_test, label_col_idx=-1)

print('\nAsserting that the filtered_X has the same columns as X_test (minus the new label column in X_test)')
assert filtered_X.shape[1] == len(X_test.columns[:-1])
labeled_wwt = pd.DataFrame(filtered_X, columns=X_test.columns[:-1])

#filter out packet features from wwt
print('\nFilter packet features out of testing for supervised calssification.')
X_test_flow = filter_wwt(labeled_wwt, split='Flow')
# final_prediction = predict_phase3(filtered_X)

# evaluate final_prediction with:
# Threshold tuned: 0.03019  (val F1=0.0497)
#               precision    recall  f1-score   support

#       Benign       0.98      0.98      0.98     25500
#       Attack       0.06      0.04      0.05       535

#     accuracy                           0.96     26035
#    macro avg       0.52      0.51      0.52     26035
# weighted avg       0.96      0.96      0.96     26035